# Sequence Lag Ridge Native Adapter v1

目标：把 `sequence_lag_ridge_mse` 从 proxy fidelity 推进为 native sequence baseline。该模型保持 lagged sequence tensor contract，将 5 日窗口展平后输入 Ridge；可复用训练/回测/analysis 逻辑全部调用 `src/`。

## 外部参考边界

- scikit-learn Ridge API: `https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html`，核对 Ridge L2 线性回归 baseline 语义。
- scikit-learn lagged feature example: `https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html`，参考时间序列滞后窗口转监督学习特征的公开做法。
- arXiv: `https://arxiv.org/abs/2205.13504`，参考时间序列 benchmark 中保留简单线性/滞后 baseline 的定位。

本实验不复制第三方代码；只确认 `sequence_lag_ridge_mse` 已使用本仓库 `build_sequence_tensor` 形成 3D lagged sequence tensor，再用 Ridge 作为可解释 native sequence baseline。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import LaggedRidgeRegressor
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'sequence_lag_ridge_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'
MODEL_NAMES = ('ridge_mse', 'sequence_lag_ridge_mse', 'lstm_mse', 'gru_mse', 'mlp_mixer_mse')

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:5])
estimator, constant_prediction = fit_estimator_for_model('sequence_lag_ridge_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('sequence_lag_ridge_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, LaggedRidgeRegressor)
assert hasattr(estimator, 'model_')
assert estimator.lookback_ == 5
assert estimator.feature_count_ == 5
assert estimator.flattened_feature_count_ == 25
assert capability['paper_fidelity'] == 'native_sequence'
assert capability['is_proxy'] is False
assert model_adapter_name('sequence_lag_ridge_mse') == 'sequence_lag_ridge_mse_v1'
assert 'sequence_lag_ridge_mse' in fidelity['native_models']
assert 'sequence_lag_ridge_mse' not in fidelity['proxy_models']
assert len(fidelity['native_models']) == 30
assert len(fidelity['proxy_models']) == 2

comparison_row = outputs.model_comparison.loc[outputs.model_comparison['model'].eq('sequence_lag_ridge_mse')].iloc[0].to_dict()
ledger_row = outputs.model_run_ledger.loc[outputs.model_run_ledger['model'].eq('sequence_lag_ridge_mse')].iloc[0].to_dict()
summary = {
    'schema_version': 'research.sequence_lag_ridge_native_adapter_v1',
    'status': 'completed',
    'model': 'sequence_lag_ridge_mse',
    'adapter': model_adapter_name('sequence_lag_ridge_mse'),
    'estimator': estimator.__class__.__name__,
    'lookback': estimator.lookback_,
    'feature_count': estimator.feature_count_,
    'flattened_feature_count': estimator.flattened_feature_count_,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'remaining_proxy_models': fidelity['proxy_models'],
    'model_fidelity_counts': fidelity['fidelity_counts'],
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'matrix_models': sorted(outputs.predictions['model'].astype(str).unique().tolist()),
    'target_model_return_pct': float(comparison_row.get('return_pct', 0.0)),
    'target_model_relative_return_pct': float(comparison_row.get('relative_return_pct', 0.0)),
    'target_model_sharpe': float(comparison_row.get('sharpe', 0.0)),
    'target_model_max_drawdown_pct': float(comparison_row.get('max_drawdown_pct', 0.0)),
    'target_model_stability_score': float(comparison_row.get('stability_score', 0.0)),
    'ledger_trained': bool(ledger_row.get('trained', False)),
    'ledger_backtested': bool(ledger_row.get('backtested', False)),
    'analysis_layers': outputs.summary.get('analysis_layers', []),
    'analysis_table_counts': outputs.summary.get('analysis_table_counts', {}),
    'external_references': [
        {
            'source_id': 'sklearn:Ridge',
            'url': 'https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html',
            'usage': '核对 Ridge L2 线性回归接口与 baseline 语义；本仓库通过 sklearn Ridge 实现。',
        },
        {
            'source_id': 'sklearn:lagged_features',
            'url': 'https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html',
            'usage': '参考滞后窗口转监督学习特征矩阵的公开方法；本仓库用 build_sequence_tensor 维护 3D sequence contract。',
        },
        {
            'source_id': 'arxiv:2205.13504',
            'url': 'https://arxiv.org/abs/2205.13504',
            'usage': '参考时间序列 benchmark 中保留简单线性/滞后 baseline 的定位。',
        },
    ],
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
        'qlib_risk_metrics': str((CARD_DIR / 'qlib_risk_metrics.csv').relative_to(REPO_ROOT)),
        'model_effect_deltas': str((CARD_DIR / 'model_effect_deltas.csv').relative_to(REPO_ROOT)),
        'model_regime_window_diagnostics': str((CARD_DIR / 'model_regime_window_diagnostics.csv').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.sequence_lag_ridge_native_adapter_v1',
 'status': 'completed',
 'model': 'sequence_lag_ridge_mse',
 'adapter': 'sequence_lag_ridge_mse_v1',
 'estimator': 'LaggedRidgeRegressor',
 'lookback': 5,
 'feature_count': 5,
 'flattened_feature_count': 25,
 'paper_fidelity': 'native_sequence',
 'is_proxy': False,
 'full_matrix_native_model_count': 30,
 'full_matrix_proxy_model_count': 2,
 'remaining_proxy_models': ['extra_trees_rank', 'ridge_ic'],
 'model_fidelity_counts': {'native_graph': 9,
  'native_library': 8,
  'native_sequence': 7,
  'native_torch': 6,
  'proxy_feature_adapter': 2},
 'matrix_card': 'research/sequence_lag_ridge_native_adapter_v1/matrix_card',
 'matrix_models': ['gru_mse',
  'lstm_mse',
  'mlp_mixer_mse',
  'ridge_mse',
  'sequence_lag_ridge_mse'],
 'target_model_return_pct': 16.627834555507714,
 'target_model_relative_return_pct': -41.490546608435764,
 'target_model_sharpe': 2.5747734784776033,
 'target_model_max_drawdown_pct': 14.0415814221946

## 结构说明

- `sequence_lag_ridge_mse` 是 native sequence baseline：输入必须是 3D sequence tensor，窗口长度默认为 5。
- estimator 记录 `lookback_`、`feature_count_`、`flattened_feature_count_`，证明训练路径使用 lagged sequence contract，而不是 plain current-row Ridge。
- 该模型用于线性基准和消融，不声称复现 LSTM/Transformer 类复杂架构。

## 中文结论

- 做了什么：给 `LaggedRidgeRegressor` 增加 native sequence 契约元数据，并把 `sequence_lag_ridge_mse` 从 proxy fidelity 升级为 `native_sequence`。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`sequence_lag_ridge_mse`、`lstm_mse`、`gru_mse`、`mlp_mixer_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=30、proxy=2；剩余 proxy 为 `ridge_ic`、`extra_trees_rank`。
- Regime/区间差异：本轮继续复用 `src/analysis` 的 window/regime/qlib 风险表，具体强弱分段以 `matrix_card/model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 `model_review_briefing.csv` 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：Lagged Ridge 是可解释线性 baseline，不是复杂序列网络；生产晋升仍需更长 walk-forward、压力窗口和对 SOXL 基准的稳健性检验。
- 下一步：剩余 proxy 为 `ridge_ic` 与 `extra_trees_rank`，需要围绕 IC/ranking objective 明确真实优化目标后再替换。